# Gene extraction 

 # Table of Contents
+ [Import](#Import_0)
+ [File processing](#File_processing_1)
	+ [Read file](#Read_file_2)
	+ [Filter and reorgineze data](#Filter_and_reorgineze_data_3)
    + [Format taxonomy](#Format_taxonomy)
+ [Organisms to search the genes ](#Organisms_to_search_the_genes_4)
+ [Extract gene](#Extract_gene_5)
+ [Result](#Result_6)


<a class="anchor" id="Import_0"></a>
# <span style="color: #1f9e89">Import</span>

In [1]:
import os
import sys
sys.path.append('../')
sys.path.append('../..')

from datetime import date


import pandas as pd
import numpy as np
from file_management import get_files_dir,check_save_file
_, INPUT_DIR, OUTPUT_DIR = get_files_dir()
from text_analysis import *


from retrieve_organism import *

In [2]:
import pandas as pd
import numpy as np
import re

from Bio.KEGG import REST
from retrieve_gene import * 

# Input and output files

## Input

In [3]:
# Articles with organism
#file = OUTPUT_DIR+'/Articles/full_text_classified_w_org_24_08_29.json'

file = OUTPUT_DIR+'/Articles/full_text_classified_w_org_25_09_19.json'
classified_org = pd.read_json(file)


In [4]:
# They only mention it is a yeast, so we search in Saccharomyces cerevisiae genes
bool_incons = (~classified_org.loc[:,'Organism'].isna()) & (classified_org.loc[:,'Full_name'].isna())

classified_org.loc[bool_incons,'Full_name'] = 'Saccharomyces cerevisiae'
def to_list(x):
    return([x])

classified_org.loc[bool_incons,'Full_name'] = classified_org.loc[bool_incons,'Full_name'].apply(to_list)

In [5]:
classified_org.loc[38831337]

Title              Selectively superior production of docosahexae...
Abstract           Schizochytrium sp. is commercially used for pr...
Journal                   Biotechnology for biofuels and bioproducts
Year                                                            2024
PMC_ID                                                    11145866.0
DOI                                       10.1186/s13068-024-02524-2
Type                                                Journal Article,
Author             [ForeName:Yana,LastName:Liu] [ForeName:Xiao,La...
Text               Introduction:\nIn this study, we decreased the...
Abstract_min       Schizochytrium sp. docosahexaenoic (DHA). Schi...
Title_min           Selectively docosahexaenoic Schizochytrium sp. .
Full_text_min      Introduction: , fas, PPTase subunits PUFA (ORF...
Organism                                                     [yeast]
Organism_Source                                            full_text
Full_name                         

In [6]:
def remove_groups(lista):
    new_lista = []
    for org in lista:
        if org.split(' ')[-1] != 'group':
            new_lista.append(org)
            
    return(new_lista)

In [7]:
bool_neg = (~classified_org.loc[:,'Organism'].isna())

In [8]:
classified_org.loc[bool_neg,'Full_name'] =classified_org.loc[bool_neg,'Full_name'].apply(remove_groups)

In [9]:
len(classified_org.loc[:,'Organism'].dropna())

13849

## Output

In [10]:
general_name = 'full_text_w_genes'

today = date.today()
today = today.strftime("%y_%m_%d")

output_file = f'{general_name}_{today}.json'
output_file


'full_text_w_genes_25_09_26.json'

<a class="anchor" id="File_processing_1"></a>
# <span style="color: #1f9e89">File processing</span>

<a class="anchor" id="Read_file_2"></a>
## <span style="color: #35b779">Read file</span>

In [11]:
len(classified_org.Full_name.dropna())

13849

In [12]:
# Remove duplicated organisms
classified_org['Full_name'] = classified_org['Full_name'].apply(lambda lista: ['yeast'] if lista == 'yeast' else (list(set(lista)) if lista is not None else None))


In [13]:
classified_org.rename(columns = {'Organism':'Old_name', 'Full_name':'Organism'}, inplace = True)

# Not reviews
classified_org = classified_org.loc[~classified_org.Type.str.lower().str.contains('review')]


In [14]:
# from the retrieve organism module
taxonomy = taxonomy.set_index('tax_id')


true_species = taxonomy.Full_name.dropna().drop_duplicates()
#full_species= true_species.str.split(' ').str[0:2].str.join(' ').drop_duplicates()
full_species = true_species.sort_values().str.split(' ').str[0:2].str.join(' ').drop_duplicates()

full_species = full_species.apply(nospecial)
full_species = full_species.str.rstrip()
full_species = full_species.drop_duplicates(keep='first')

full_species = full_species.str.strip('.')
tax_species_dict = {v: k for k, v in full_species.items()}


In [15]:
true_species

tax_id
7                    Azorhizobium caulinodans
9                         Buchnera aphidicola
11                        Cellulomonas gilvus
14                  Dictyoglomus thermophilum
17               Methylophilus methylotrophus
                          ...                
2995172                 Pseudomonas sp. S1Bt3
2995173                 Pseudomonas sp. S1Bt7
2995174                Pseudomonas sp. S1Bt42
2995225     Sphingobacterium sp. UT-1RO-CII-1
2995235    Scleroderma venenatum (nom. inval.
Name: Full_name, Length: 582327, dtype: object

In [16]:
true_genus = taxonomy.Genus.dropna().drop_duplicates()
genus_list = true_genus.str.split('/').explode().drop_duplicates()

tax_genus_dict = {v: k for k, v in genus_list.items()}


In [17]:
tax_species_dict['Escherichia coli'] == 562

True

In [18]:
# Genus
tax_species_dict['Komagataella sp'] = '4922'
tax_species_dict['Phlebia sp'] = '5307'

# Species 

tax_species_dict['Populus tremula'] = '113636'

tax_species_dict['Cystobacter sp'] = '1965334'
tax_species_dict['Cyanothece sp'] = '43988'

tax_species_dict['Klebsiella species'] = '570'
tax_species_dict['Arthrospira sp']= '35824'
tax_species_dict['Gluconobacter sp'] = '1876758'
tax_species_dict['Gluconacetobacter sp'] = '89583'
tax_species_dict['Halomonas sp']='2745'

tax_species_dict['Agrobacterium aurantiacum']='357'
tax_species_dict['Arthrospira sp']='35823'
tax_species_dict['Rhizopus sp']='4842'
tax_species_dict['Fusarium solani']='169388'
tax_species_dict['Candida sp']='1853550'
tax_species_dict['Actinoplanes sp']='1865'
tax_species_dict['Saccharomonospora sp'] ='1851'
#halomonas genre
tax_species_dict['Halomonas bluephagenesis'] = '2745'
# Geobacillus genre
tax_species_dict['Geobacillus sp'] = '129337'

#genus
tax_species_dict['Rhodotorula sp'] = '5533'
tax_species_dict['Neocallimastix sp'] = '4756'
tax_species_dict['Phomopsis sp'] = '34399'
tax_species_dict['Hydrogenivirga sp'] =  '246263'
tax_species_dict['Lyngbya sp'] = '28073'
tax_species_dict['Chloroflexi bacterium'] ='200795'
tax_species_dict['Lysinibacillus sp'] = '400634'
tax_species_dict['Pavlova sp'] = '2831'
tax_species_dict['Terrabacteria group'] = '1783272'

#species
tax_species_dict['Hydrogenobaculum sp'] = '2053560'
tax_species_dict['Leifsonia sp'] = '1870902'
tax_species_dict['Jeotgalicoccus sp'] = '1871619'
tax_species_dict['Aurantiochytrium sp'] = '1689870'
tax_species_dict['Pandoraea sp'] = '1883445'
tax_species_dict['Nonomuraea sp'] = '1883105'
tax_species_dict['Nonomuraea terrinata'] = '83681'
tax_species_dict['Geobacter sulfurreducens'] = '35554'
tax_species_dict['Neotyphodium sp'] = '2879987'

#unclass
tax_species_dict['Mesorhizobium sp'] ='325217'
tax_species_dict['Methanosaeta sp'] = '2620051'
tax_species_dict['Pseudoxanthomonas sp'] = '2645906'
tax_species_dict['Synechocystis pcc']='1148'

<a class="anchor" id="Filter_and_reorgineze_data_3"></a>
## <span style="color: #35b779">Filter and reorgineze data</span>

In [19]:
#org_prod is exploded,
#classified_org is not exploded

In [20]:
# Remove problematic phrases
classified_org['Abstract_min'] = classified_org['Abstract_min'].replace(REPLACEMENTS, regex=True)
classified_org['Title_min'] = classified_org['Title_min'].replace(REPLACEMENTS, regex=True)
classified_org['Full_text_min'] = classified_org['Full_text_min'].replace(REPLACEMENTS, regex=True)

# Remove verbs
classified_org.loc[:,'Abstract_min'] = classified_org.loc[:,'Abstract_min'].apply(remove_verbs, case_relevant=True)
classified_org.loc[:,'Title_min'] = classified_org.loc[:,'Title_min'].apply(remove_verbs, case_relevant=True)
classified_org['Full_text_min'] = classified_org['Full_text_min'].map(
    lambda x: remove_verbs(x, case_relevant=True) if pd.notna(x) else np.nan
)

# Remove nouns
classified_org.loc[:,'Abstract_min'] = classified_org.loc[:,'Abstract_min'].map(remove_noun)
classified_org.loc[:,'Title_min'] = classified_org.loc[:,'Title_min'].map(remove_noun)
#classified_org.loc[:,'Full_text_min'] = classified_org.loc[:,'Full_text_min'].map(remove_noun)
classified_org.loc[:, 'Full_text_min'] = classified_org['Full_text_min'].map(
    lambda x: remove_noun(x) if pd.notna(x) else np.nan
)

In [21]:
#Apply same logic as when we removed organisms, and remove 

In [22]:
from collections import Counter
import re


In [23]:

words_to_remove = set(['escherichia', 'coli', 'cerevisiae','corynebacterium','efficient',
                   'sp','bacillus','enhanced','yarrowia','streptomyces','development'])


In [24]:

words_to_remove.update(['dehydrogenase', 'including', 'e', 's','c','constructed',
                   'achieved','overexpression','producing','highest','strategies','compound','strategy',
                   'revealed','provide','furthermore','provide','enhance','demonstrate','p',
                   'finally','chemical','without','express','optimiz','pyruvate'])


In [25]:

words_to_remove.update(['et', 'al', 'fig', 'mm','ml','added','generate','metabolite'
                   'a','x','wild-type','primers','media','incubated','b','studi','lb'
                   'sample','supplementary','mutant','supplement','indicat','per','would',
                   'therefore','m','wt','experiment','carrie','amplifi','h','glutamicum',
                       'predict'])


In [27]:
words_to_remove.update(fake_genes)

In [28]:
words_to_remove = set(words_to_remove)
pattern = r'\b(' + '|'.join(words_to_remove) + r')e?s?d?i?n?g?\b'

# Remove words and clean up extra spaces
classified_org.loc[:, 'Abstract_min'] = (
    classified_org.loc[:, 'Abstract_min']
    .str.replace(pattern, '', regex=True, flags=re.IGNORECASE)
    .str.replace(r'\s+', ' ', regex=True)  # Replace multiple spaces with single space
    .str.strip()  # Remove leading/trailing spaces
)
# Remove words and clean up extra spaces
classified_org.loc[:, 'Title_min'] = (
    classified_org.loc[:, 'Title_min']
    .str.replace(pattern, '', regex=True, flags=re.IGNORECASE)
    .str.replace(r'\s+', ' ', regex=True)  # Replace multiple spaces with single space
    .str.strip()  # Remove leading/trailing spaces
)
# Remove words and clean up extra spaces
classified_org.loc[:, 'Full_text_min'] = (
    classified_org.loc[:, 'Full_text_min']
    .str.replace(pattern, '', regex=True, flags=re.IGNORECASE)
    .str.replace(r'\s+', ' ', regex=True)  # Replace multiple spaces with single space
    .str.strip()  # Remove leading/trailing spaces
)


In [29]:
# Format the file to search
classified_org_format_title = format_file(classified_org, 'Title_min')
classified_org_format_abstract = format_file(classified_org, 'Abstract_min')
classified_org_format_full_text = format_file(classified_org.dropna(subset='Full_text_min'), 'Full_text_min')


<a class="anchor" id="Organisms_to_search_the_genes_4"></a>
# <span style="color: #1f9e89">Organisms to search the genes </span>

## Change some organisms names to synonyms

In [31]:
#org_prod.Organism = org_prod.Organism.str.replace(' spp',' sp').str.replace(' species',' sp')

## Extract tax ID

In [32]:
org_prod = classified_org.explode('Organism')
org_prod = org_prod.dropna(subset=['Organism'])
org_prod = org_prod[org_prod['Organism'] != 'C roseus']
org_prod.Organism = org_prod.Organism.replace(ORG_SYNONYM)


In [33]:
org_prod.loc[:,'Tax_id_species'] = org_prod.Organism.str.capitalize().map(tax_species_dict)
org_prod.loc[:,'Tax_id_genus'] = org_prod.Organism.str.capitalize().str.split(' ').str[0].map(tax_genus_dict)

In [34]:
org_prod

,Title,Abstract,Journal,Year,PMC_ID,DOI,Type,Author,Text,Abstract_min,Title_min,Full_text_min,Old_name,Organism_Source,Organism,Tax_id_species,Tax_id_genus
10618204,Increased production of zeaxanthin and other p...,The psbAII locus was used as an integration pl...,Applied and environmental microbiology,2000,91786.0,10.1128/AEM.66.1.64-72.2000,"Journal Article,Research Support, U.S. Gov't, ...","[ForeName:D,LastName:Lagarde] [ForeName:L,Last...",None,psbAII overexpress Synechocystis . PCC psbAII ...,pigments techniques Synechocystis . PCC .,NaN,[Synechocystis PCC],title,Synechocystis PCC,1148,1143
10618209,Expression of Alcaligenes eutrophus flavohemop...,Expression of the vhb gene encoding hemoglobin...,Applied and environmental microbiology,2000,91791.0,10.1128/AEM.66.1.98-104.2000,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:A D,LastName:Frey] [ForeName:J E,Las...",None,vhb Vitreoscilla . (VHb) organisms microaerobi...,Alcaligenes eutrophus flavohemoprotein Vitreos...,NaN,[Escherichia coli],title,Escherichia coli,562,562
10649237,Altered regulation of pyruvate kinase or co-ov...,Glycolytic fluxes in resting Escherichia coli ...,Biotechnology and bioengineering,2000,NaN,10.1002/(sici)1097-0290(20000305)67:5<623::aid...,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:M,LastName:Emmerling] [ForeName:J E,...",None,Glycolytic resting kinases (Pyk) stearothermop...,Altered co- phosphofructokinase glycolytic res...,NaN,[Escherichia coli],title,Escherichia coli,562,562
10649449,Cloning and characterization of the Yarrowia l...,The squalene synthase (SQS) gene encodes a key...,"Yeast (Chichester, England)",2000,NaN,10.1002/(SICI)1097-0061(200002)16:3<197::AID-Y...,"Journal Article,Research Support, Non-U.S. Gov...","[ForeName:S,LastName:Merkulov] [ForeName:F,Las...",None,"squalene (SQS) encodes , farnesyl-diphosphate ...",Cloning lipolytica squalene (SQS1) erg9 .,NaN,"[Yarrowia lipolytica, Saccharomyces cerevisiae]",title,Saccharomyces cerevisiae,4932,4931
10649449,Cloning and characterization of the Yarrowia l...,The squalene synthase (SQS) gene encodes a key...,"Yeast (Chichester, England)",2000,NaN,10.1002/(SICI)1097-0061(200002)16:3<197::AID-Y...,"Journal Article,Research Support, Non-U.S. Gov...","[ForeName:S,LastName:Merkulov] [ForeName:F,Las...",None,"squalene (SQS) encodes , farnesyl-diphosphate ...",Cloning lipolytica squalene (SQS1) erg9 .,NaN,"[Yarrowia lipolytica, Saccharomyces cerevisiae]",title,Yarrowia lipolytica,4952,4952
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40572208,Heterologous Expression of the Nitrogen-Fixing...,Microbially mediated biological nitrogen fixat...,Microorganisms,2025,12195393.0,10.3390/microorganisms13061320,"Journal Article,","[ForeName:Xiuling,LastName:Wang] [ForeName:Shi...",Introduction:\nThis article reports the first ...,"Microbially . , hindered complexities manipula...",Heterologous Nitrogen-Fixing Cluster <i>Paenib...,Introduction: reports nif .polymyxa .subtilis....,"[Paenibacillus polymyxa, Bacillus subtilis]",title,Paenibacillus polymyxa,1406,1401
40572208,Heterologous Expression of the Nitrogen-Fixing...,Microbially mediated biological nitrogen fixat...,Microorganisms,2025,12195393.0,10.3390/microorganisms13061320,"Journal Article,","[ForeName:Xiuling,LastName:Wang] [ForeName:Shi...",Introduction:\nThis article reports the first ...,"Microbially . , hindered complexities manipula...",Heterologous Nitrogen-Fixing Cluster <i>Paenib...,Introduction: reports nif .polymyxa .subtilis....,"[Paenibacillus polymyxa, Bacillus subtilis]",title,Bacillus subtilis,1423,1390
40577193,Structural characterization and dynamics of Ad...,"<i>Acetivibrio thermocellus</i>, a cellulolyti...",eLife,2025,NaN,10.7554/eLife.96966,"Journal Article,","[ForeName:Samantha J,LastName:Ziegler] [ForeNa...",None,"<i>Acetivibrio thermocellus</i>, cellulolytic ...",Structural AdhE ultrastructures <i>Acetivibrio...,NaN,[Acetivibrio thermocellus],title,Acetiv

In [35]:
tax_species_ids  = org_prod.loc[:,['Organism','Tax_id_species']].set_index('Organism').dropna().drop_duplicates()
tax_species_ids = tax_species_ids.astype({'Tax_id_species':'int'}).astype({'Tax_id_species':'str'}).to_dict()['Tax_id_species']

tax_genus_ids  = org_prod.loc[:,['Organism','Tax_id_genus']].set_index('Organism').dropna().drop_duplicates()
tax_genus_ids.index = tax_genus_ids.index.str.split(' ').str[0]
tax_genus_ids = tax_genus_ids.astype({'Tax_id_genus':'int'}).astype({'Tax_id_genus':'str'}).to_dict()['Tax_id_genus']



## Get genes data from UniProt

In [36]:
org_prod[org_prod.Organism.str.contains('Pseudoxanthomonas')]

,Title,Abstract,Journal,Year,PMC_ID,DOI,Type,Author,Text,Abstract_min,Title_min,Full_text_min,Old_name,Organism_Source,Organism,Tax_id_species,Tax_id_genus
33995910,Widespread distribution of <i>hmf</i> genes in...,Furans represent a class of promising chemical...,Computational and structural biotechnology jou...,2021,8091172.0,10.1016/j.csbj.2021.04.017,"Journal Article,","[ForeName:Raúl A,LastName:Donoso] [ForeName:Fa...",Introduction:\nIn this work we performed an ex...,"Furans represent , since constitute intermedia...",Widespread <i>hmf</i> Proteobacteria reveals -...,Introduction: hmf/psf proteobacterial genomes ...,"[Raoultella ornithinolytica, Azospirillum oryz...",full_text,Pseudoxanthomonas composti,2137479,69284


Manual curation 

check why this species are not correct and also add a way to get the genus when the species is not found 

In [37]:
tax_species_ids

{'Synechocystis PCC': '1148',
 'Escherichia coli': '562',
 'Saccharomyces cerevisiae': '4932',
 'Yarrowia lipolytica': '4952',
 'Saccharopolyspora erythraea': '1836',
 'Pseudomonas putida': '303',
 'Bacillus subtilis': '1423',
 'Streptomyces galilaeus': '33899',
 'Komagataella pastoris': '4922',
 'Priestia megaterium': '1404',
 'Lactococcus lactis': '1358',
 'Cupriavidus necator': '106590',
 'Anabaena cylindrica': '1165',
 'Corynebacterium ammoniagenes': '1697',
 'Streptomyces lividans': '1916',
 'Lactobacillus helveticus': '1587',
 'Streptomyces collinus': '42684',
 'Streptomyces verticillus': '29309',
 'Lacticaseibacillus casei': '1582',
 'Pseudomonas fluorescens': '294',
 'Bacillus licheniformis': '1402',
 'Klebsiella oxytoca': '571',
 'Streptomyces maritimus': '115828',
 'Rhodococcus erythropolis': '1833',
 'Streptomyces glaucescens': '1907',
 'Sorangium cellulosum': '56',
 'Streptomyces coelicolor': '1902',
 'Aspergillus nidulans': '162425',
 'Streptomyces clavuligerus': '1901',
 

In [38]:
# Genus
tax_species_ids['Komagataella sp'] = '4922'
tax_species_ids['Phlebia sp'] = '5307'

# Species 

tax_species_ids['Populus tremula'] = '113636'

tax_species_ids['Cystobacter sp'] = '1965334'
tax_species_ids['Cyanothece sp'] = '43988'

tax_species_ids['Klebsiella species'] = '570'
tax_species_ids['Arthrospira sp']= '35824'
tax_species_ids['Gluconobacter sp'] = '1876758'
tax_species_ids['Gluconacetobacter sp'] = '89583'
tax_species_ids['Halomonas sp']='2745'

tax_species_ids['Agrobacterium aurantiacum']='357'
tax_species_ids['Arthrospira sp']='35823'
tax_species_ids['Rhizopus sp']='4842'
tax_species_ids['Fusarium solani']='169388'
tax_species_ids['Candida sp']='1853550'
tax_species_ids['Actinoplanes sp']='1865'
tax_species_ids['Saccharomonospora sp'] ='1851'
#halomonas genre
tax_species_ids['Halomonas bluephagenesis'] = '2745'
# Geobacillus genre
tax_species_ids['Geobacillus sp'] = '129337'

#genus
tax_species_ids['Rhodotorula sp'] = '5533'
tax_species_ids['Neocallimastix sp'] = '4756'
tax_species_ids['Phomopsis sp'] = '34399'
tax_species_ids['Hydrogenivirga sp'] =  '246263'
tax_species_ids['Lyngbya sp'] = '28073'
tax_species_ids['Chloroflexi bacterium'] ='200795'
tax_species_ids['Lysinibacillus sp'] = '400634'
tax_species_ids['Pavlova sp'] = '2831'
tax_species_ids['Terrabacteria group'] = '1783272'

#species
tax_species_ids['Hydrogenobaculum sp'] = '2053560'
tax_species_ids['Leifsonia sp'] = '1870902'
tax_species_ids['Jeotgalicoccus sp'] = '1871619'
tax_species_ids['Aurantiochytrium sp'] = '1689870'
tax_species_ids['Pandoraea sp'] = '1883445'
tax_species_ids['Nonomuraea sp'] = '1883105'
tax_species_ids['Nonomuraea terrinata'] = '83681'
tax_species_ids['Geobacter sulfurreducens'] = '35554'
tax_species_ids['Neotyphodium sp'] = '2879987'

#unclass
tax_species_ids['Mesorhizobium sp'] ='325217'
tax_species_ids['Methanosaeta sp'] = '2620051'
tax_species_ids['Pseudoxanthomonas sp'] = '2645906'
tax_species_ids['Synechocystis pcc']='1148'
tax_species_ids['Synechocystis PCC']='1148'

#Holophaga sp
tax_species_ids['Holophaga foetida']= '2582906'

download per genus..

In [70]:
def download_genes_per_organism(top_organisms):
    """
    Downloads gene information for a set of top organisms.

    Parameters
    ----------
    top_organisms : dict
        A dictionary containing the names of the top organisms as keys and their corresponding
        taxonomy IDs as values.

    Returns
    -------
    tuple
        A tuple containing three dictionaries: (1) a dictionary of Pandas series where each series 
        contains the gene names for an organism, (2) a dictionary of Pandas dataframes where each 
        dataframe contains the synonyms for the genes of an organism, and (3) a dictionary of Pandas 
        dataframes where each dataframe contains the full UniProt information for the genes of an organism.

    Raises
    ------
    Exception
        If the data cannot be downloaded or processed for any of the specified organisms, an exception is raised.

    Notes
    -----
    This function downloads gene information for the top organisms specified in the input dictionary.
    It first tries to read pre-existing files containing the gene information for each organism, and if
    those files do not exist, it downloads the information from the UniProt website. The gene information 
    is saved in three different formats: a Pandas series containing only the gene names, a Pandas 
    dataframe containing the synonyms for the genes, and a Pandas dataframe containing the full UniProt 
    information for the genes. The resulting dictionaries contain the gene information for each organism.
    """
    series = {}
    series_synonyms = {}
    full_table={}
    faulty = {}
    # Loop through each organism and download or read the gene information
    for organism,taxid in top_organisms.items():
        
        file_name = organism.replace(' ', '_')+'.csv'
        
        try:
            # Full_table
            #genes_pd = pd.read_csv('./Full_UniProt/'+file_name, dtype=str)
            formated_table = pd.read_csv('../Genes/'+file_name, dtype=str)
            full_table[organism] = formated_table
            
        except:
            print('Fail1',file_name)
            try:
                print('holi')
                genes_pd = pd.read_csv('../Full_UniProt/'+file_name, dtype=str)
            except:
                print('Fail2')
                # If pre-existing files do not exist, download gene information from UniProt website
                #print('no file, lets retrieve it')
                url = generate_url(taxid, reviewed=False)
                print(url)
                genes_pd = retrieve_data(url)
                if genes_pd.empty:
                    print(f'\nNo reviewed UniProt info for {organism} with taxid: {taxid}')
                    url = generate_url(taxid, reviewed=False)
                    genes_pd = retrieve_data(url)
                    if genes_pd.empty:
                        print(f'No UniProt info for {organism} with taxid: {taxid}')
                        faulty[organism] = taxid
                        continue
                    else:
                        print('Using unreviewed data instead')
                        genes_pd.to_csv('../Full_UniProt/'+file_name, index=False)
                else:
                    genes_pd.to_csv('../Full_UniProt/'+file_name, index=False)

                #genes_filter.sort()
                # Save gene information to files


            #KEGG id (full table, maybe add it to synonyms or genes?)
            if genes_pd.columns[0] == 'Error messages':
                print(genes_pd.iloc[0,0])
                faulty[organism] = taxid
                continue
            try:
                formated_table, valid_genes = format_all_genes_new(genes_pd)
            except:
                print('I am failing')
                return(genes_pd)
            if not valid_genes:
                print(f'No gene names in uniprot for the {organism}')
                faulty[organism] = taxid
                continue
            full_table[organism] = formated_table
            formated_table.to_csv('../Genes/'+file_name, index=False)
            
    
    # Swap the keys and values of All_gene_names_x
    #swap_dict = {v: k for k, v in indexed_concatenated_df.All_gene_names_x.to_dict().items()}
        
    return(full_table,faulty)
# Add organisms to fake genes
# Add restriction enzymes
# Remove also cas genes?


def get_genes_from_text(sentence, organism, organism_genes):
    """
    Extracts gene names from a given sentence that match a specific organism's genes.

    Parameters:
    -----------
    sentence : pd.Series
        A pandas Series of strings representing the input sentence(s) to extract gene names from.
    organism : str
        A string representing the name of the organism for which gene names will be extracted.
    organism_genes : dict
        A dictionary where keys are the names of organisms and values are pandas Series 
        of gene names for each respective organism.

    Returns:
    --------
    A list of strings representing gene names that were found in the input sentence(s) 
    and match the gene names for the specified organism. If no gene names were found, 
    returns None.

    Raises:
    -------
    KeyError: If the organism name provided is not found in the organism_genes dictionary.
    """
    
    if organism in organism_genes.keys():
        sentence  = sentence.dropna().drop_duplicates()
        sentence = sentence.loc[sentence.str.len()>2]
        try:
            organism_data = organism_genes[organism].groupby('superset_genes').agg(sum).reset_index()
        except:
            print(organism_genes[organism])
        
        genes_found = search_gene(sentence, organism_data)
        
        if  genes_found[0][0] != 'try_again':
            return(genes_found)
        else:
            # Instead of searching araC, search AraC
            first_cap = organism_data.superset_genes.str[0].str.upper()+organism_data.superset_genes.str[1:]
            organism_first_cap = organism_data.loc[organism_data.superset_genes != first_cap]
            
            genes_found = search_gene(sentence, organism_first_cap)
            
            if  genes_found[0][0] != 'try_again':
                return(genes_found)
            else:
                # Searh lower just in case
                lower_cap = organism_data.superset_genes.str.lower()
                organism_lower_cap = organism_data.loc[organism_data.superset_genes != lower_cap]
                
                sentence = sentence.str.lower()
                
                genes_found = search_gene(sentence, organism_lower_cap)
                
                if  genes_found[0][0] != 'try_again':
                    return(genes_found)
    else:
        
        print(f'Organism {organism} not found')
        
        

In [49]:

def format_all_genes_new(genes_pd: pd.DataFrame) -> pd.DataFrame:
    """
    Cleaner reimplementation of format_all_genes.
    Keeps the same output schema/format, now also handling EC numbers.
    """

    # --- Step 1: Prepare input ---
    df = genes_pd.copy()
    try:
        df = df.set_index("Entry")
    except:
        print(df)

    # Merge main + synonym names
    df["All_gene_names"] = (
        df["Gene Names"].fillna("").astype(str) + " " +
        df["Gene Names (synonym)"].fillna("").astype(str)
    ).str.strip()

    # --- NEW: Drop rows with no gene names at all ---
    df = df[df["All_gene_names"] != ""]
    if df.empty:
        return None, False   # flag = True → no valid gene names    
    
    # Keep only relevant cols
    df = df[["All_gene_names", "KEGG", "EC number"]].drop_duplicates()

    # Tokenize and clean gene names
    df["All_gene_names"] = df["All_gene_names"].str.strip().str.split()
    df = df.explode("All_gene_names").drop_duplicates()
    df = df[df["All_gene_names"].str.len() > 2]
    df["All_gene_names"] = df["All_gene_names"].str.replace(r"['()]", "", regex=True)
    df = df.drop_duplicates().reset_index()  # restore Entry as column

    # --- Step 2: Groupings ---

    # Helper: split EC numbers into sets
    def split_ec(series):
        out = set()
        for val in series.dropna():
            for ec in str(val).split(";"):
                ec = ec.strip()
                if ec:
                    out.add(ec)
        return out

    # For each gene
    gene_groups = df.groupby("All_gene_names").agg({
        "Entry": set,
        "KEGG": lambda x: set(x.dropna()),
        "EC number": split_ec
    }).rename(columns={
        "Entry": "Entry_Set",
        "KEGG": "KEGG_genes_set",
        "EC number": "EC_genes_set"
    })

    # For each entry
    entry_groups = df.groupby("Entry").agg({
        "All_gene_names": set,
        "KEGG": lambda x: set(x.dropna()),
        "EC number": split_ec
    }).rename(columns={
        "All_gene_names": "Genes_set",
        "KEGG": "KEGG_entry_set",
        "EC number": "EC_entry_set"
    })

    # Merge back into df
    merged = df.merge(gene_groups, left_on="All_gene_names", right_index=True)
    merged = merged.merge(entry_groups, on="Entry")

    # Supersets
    superset_entry = merged.groupby("Entry")["Entry_Set"].apply(lambda sets: set.union(*sets)).rename("superset_entry")
    superset_genes = merged.groupby("All_gene_names")["Genes_set"].apply(lambda sets: set.union(*sets)).rename("superset_genes")

    merged = merged.merge(superset_entry, on="Entry")
    merged = merged.merge(superset_genes, left_on="All_gene_names", right_index=True)

    # KEGG + EC supersets
    merged["KEGG_union_superset"] = merged.apply(
        lambda row: row["KEGG_entry_set"].union(row["KEGG_genes_set"]), axis=1
    )
    merged["EC_union_superset"] = merged.apply(
        lambda row: row["EC_entry_set"].union(row["EC_genes_set"]), axis=1
    )

    # --- Step 3: Format output ---
    values = merged.loc[:, ["superset_entry", "superset_genes", "KEGG_union_superset", "EC_union_superset"]]

    values["superset_entry"] = values["superset_entry"].apply(lambda x: "_".join(sorted(list(x))))
    values["superset_genes"] = values["superset_genes"].apply(list)

    # KEGG: same as old function (concatenate without delimiter)
    values["KEGG_union_superset"] = values["KEGG_union_superset"].apply(
        lambda x: "".join(sorted([v for v in x if v]))
    )

    # EC: always join with ";" for clarity, and ensure trailing ";"
    values["EC_union_superset"] = values["EC_union_superset"].apply(
        lambda x: ";".join(sorted([v for v in x if v])) + (";" if x else "")
    )

    # One row per gene
    final_table = values.explode("superset_genes").drop_duplicates()

    # Legacy behavior: group and concat with sum
    return final_table.groupby("superset_genes").agg("sum").reset_index(), True

In [ ]:
test = download_genes_per_organism(tax_species_ids) # small_set


In [53]:
download_per_species, faulty_2 = test

In [ ]:
download_per_species, faulty_2 = download_genes_per_organism(tax_species_ids) # small_set


In [54]:
faulty_2

{'Aspergillus fumigatus': '41122',
 'beta proteobacterium': '134206',
 'Corynebacterium crenatum': '168810',
 'Chromohalobacter salexigens': '158080',
 'Asteromyces cruciatus': '1407621',
 'Nocardia tartaricans': '1165097',
 'Macrocystidia cucumis': '182055',
 'Picrophilus torridus': '82076',
 'Acetobacterium fimetarium': '52691',
 'Terrisporobacter mayombei': '1541',
 'Streptomyces albulus': '68570',
 'Chaetoceros gracilis': '184592',
 'Leuconostoc garlicum': '255248',
 'Porphyridium cruentum': '2891951',
 'Competibacter phosphatis': '221280',
 'Ruminococcaceae bacterium': '742724',
 'Paracoccus carotinifaciens': '65151',
 'Streptomyces kebangsaanensis': '864058',
 'Puccinia emaculata': '582727',
 'Nevskia ramosa': '64002',
 'Bacillus velezensis': '207880',
 'Actinoalloteichus cyanogriseus': '65497',
 'Streptomyces hydrogenans': '1873719',
 'Antrodia cinnamomea': '279009',
 'Dichloromethanomonas elyunquensis': '1755312',
 'Luteimonas huabeiensis': '1244513',
 'Nitratireductor aquimari

In [55]:
coso = download_per_species


<a class="anchor" id="Result_6"></a>
# <span style="color: #1f9e89">Result</span>

In [56]:
complete_info  = pd.concat([classified_org_format_title,
                            classified_org_format_abstract,
                            classified_org_format_full_text
                           ],axis=1)

In [57]:
complete_info.head()

,0,1,2,3,4,5,6,7,8,9,...,2301,2302,2303,2304,2305,2306,2307,2308,2309,2310
10618204,pigments,techniques,Synechocystis,PCC,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10618209,Alcaligenes,eutrophus,flavohemoprotein,Vitreoscilla,hemoglobin,reductase,hypoxic,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10631776,Environmental,biotechnology,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10649237,Altered,phosphofructokinase,glycolytic,resting,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10649449,Cloning,lipolytica,squalene,SQS1,erg9,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
complete_info  = pd.concat([classified_org_format_title,
                            classified_org_format_abstract,
                            classified_org_format_full_text
                           ],axis=1)

# Merge by index (expanded)
expanded = complete_info.merge(org_prod.loc[:,['Organism']], left_index=True, right_index=True)
expanded.head(30)

In [58]:
org_prod.loc[38831337]

Title              Selectively superior production of docosahexae...
Abstract           Schizochytrium sp. is commercially used for pr...
Journal                   Biotechnology for biofuels and bioproducts
Year                                                            2024
PMC_ID                                                    11145866.0
DOI                                       10.1186/s13068-024-02524-2
Type                                                Journal Article,
Author             [ForeName:Yana,LastName:Liu] [ForeName:Xiao,La...
Text               Introduction:\nIn this study, we decreased the...
Abstract_min       Schizochytrium . docosahexaenoic (DHA). Schizo...
Title_min             Selectively docosahexaenoic Schizochytrium . .
Full_text_min      Introduction: , fas, PPTase PUFA (ORFA, ORFB, ...
Old_name                                                     [yeast]
Organism_Source                                            full_text
Organism                          

In [59]:
org_prod.Organism.loc[38831337]

'Saccharomyces cerevisiae'

In [60]:
# Merge by index (expanded)
expanded = complete_info.merge(org_prod.loc[:,['Organism']], left_index=True, right_index=True)
expanded.head()

,0,1,2,3,4,5,6,7,8,9,...,2302,2303,2304,2305,2306,2307,2308,2309,2310,Organism
10618204,pigments,techniques,Synechocystis,PCC,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Synechocystis PCC
10618209,Alcaligenes,eutrophus,flavohemoprotein,Vitreoscilla,hemoglobin,reductase,hypoxic,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Escherichia coli
10649237,Altered,phosphofructokinase,glycolytic,resting,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Escherichia coli
10649449,Cloning,lipolytica,squalene,SQS1,erg9,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Saccharomyces cerevisiae
10649449,Cloning,lipolytica,squalene,SQS1,erg9,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yarrowia lipolytica
10653745,poly,hydroxyalkanoic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Escherichia coli
10662693,Formation,complexes,picromycin,oleandomycin,polyketide,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Saccharopolyspora erythraea
10662693,Formation,complexes,picromycin,oleandomycin,polyketide,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Escherichia coli
10708651,p450,cam,CYP101,polycyclic,hydrocarbons,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Pseudomonas putida
10742205,Properties,poly,hydroxyalkanoates,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Escherichia coli


In [61]:
expanded.loc[38831337]

0                        Selectively
1                    docosahexaenoic
2                     Schizochytrium
3                                NaN
4                                NaN
                      ...           
2307                             NaN
2308                             NaN
2309                             NaN
2310                             NaN
Organism    Saccharomyces cerevisiae
Name: 38831337, Length: 2417, dtype: object

In [62]:
full = expanded

In [63]:
get_genes_from_text(full.iloc[32,0:-1],full.iloc[32,-1],coso)

/var/folders/v6/y2v6wwn93yj5vvhl558fx2zc0000gn/T/ipykernel_92025/1503431549.py:133: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  organism_data = organism_genes[organism].groupby('superset_genes').agg(sum).reset_index()


array([['GDH1', 'A0A6A5Q4B4_P07262A0A6A5Q4B4_P07262', 'sce:YOR375C;',
        '1.4.1.4;'],
       ['GLN1', 'A0A6A5Q6E7_P32288A0A6A5Q6E7_P32288', 'sce:YPR035W;',
        '6.3.1.2;6.3.1.2;'],
       ['GLT1', 'A0A8H8UMC6_Q12680A0A8H8UMC6_Q12680', 'sce:YDL171C;',
        '1.4.1.14;1.4.1.14;']], dtype=object)

In [64]:
full.iloc[32,0:-1].str.lower().dropna()

0         optimization
1         assimilation
0          originating
1     biotechnological
2               amount
3           byproducts
4                  one
5             glycerol
6            reoxidize
7             evaluate
8              whether
9         cultivations
10                gln1
11                glt1
12                gdh1
13           dependent
14             deleted
15              hereby
16           consuming
17        oxoglutarate
18         substituted
19            consumed
20                tn19
21                gdh1
22               pgk1p
23               value
24             suggest
25        circumvented
26          activities
27               gln1p
28               glt1p
29                more
30                thus
31              verify
32            proposed
Name: 10935936, dtype: object

---------

In [ ]:
for organism in coso:
    coso[organism] = remove_uninformative_genes(coso[organism])

In [75]:
def get_genes_from_text(sentence, organism, organism_genes):
    """
    Extracts gene names from a given sentence that match a specific organism's genes.

    Parameters:
    -----------
    sentence : pd.Series
        A pandas Series of strings representing the input sentence(s) to extract gene names from.
    organism : str
        A string representing the name of the organism for which gene names will be extracted.
    organism_genes : dict
        A dictionary where keys are the names of organisms and values are pandas Series 
        of gene names for each respective organism.

    Returns:
    --------
    A list of strings representing gene names that were found in the input sentence(s) 
    and match the gene names for the specified organism. If no gene names were found, 
    returns None.

    Raises:
    -------
    KeyError: If the organism name provided is not found in the organism_genes dictionary.
    """
    
    if organism in organism_genes.keys():
        
        sentence  = sentence.dropna().drop_duplicates()
        sentence = sentence.loc[sentence.str.len()>2]
        organism_data = organism_genes[organism]
        #organism_data = organism_genes[organism].groupby('superset_genes').agg(sum).reset_index()
        try:
            organism_data.superset_genes
        except:
            print(organism_data)
            print(organism)
        genes_found = search_gene(sentence, organism_data)
        #print('miau')

        if  genes_found[0][0] != 'Try_again':
            return(genes_found)
        
        #print('try other')
        # Instead of searching araC, search AraC
        try:
            organism_first_cap, genes_found = search_cap_variants(sentence, organism_data)

            if  genes_found[0][0] != 'Try_again':
                return(genes_found)
        except:
            None
            #print('f1')
        #print('try other2')
        # Instead of searching araC, search ARAC
        try:
            organism_upper, genes_found = search_upper_variants(sentence, organism_data, organism_first_cap)

            if  genes_found[0][0] != 'Try_again':
                return(genes_found)
        except:
            None
            #print('f2')
        try:
            # if araC not present, change things like ARAC into araC and search
            organism_last_cap, genes_found = search_last_cap_variants(sentence, organism_data)
            #print('mauau')
            if  genes_found[0][0] != 'Try_again':
                return(genes_found)
        except:
            None
            #print('f3')
    else:
        
        print(f'Organism {organism} not found')
        
        

In [67]:
title_data = classified_org_format_title.merge(org_prod.loc[:,['Organism']], left_index=True, right_index=True)
abstract_data = classified_org_format_abstract.merge(org_prod.loc[:,['Organism']], left_index=True, right_index=True)

In [118]:
fulltext_data = classified_org_format_full_text.merge(org_prod.loc[:,['Organism']], left_index=True, right_index=True)

In [82]:
title_data.loc[:,'Genes']

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,Organism
10618204,pigments,techniques,Synechocystis,PCC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Synechocystis PCC
10618209,Alcaligenes,eutrophus,flavohemoprotein,Vitreoscilla,hemoglobin,reductase,hypoxic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Escherichia coli
10649237,Altered,phosphofructokinase,glycolytic,resting,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Escherichia coli
10649449,Cloning,lipolytica,squalene,SQS1,erg9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Saccharomyces cerevisiae
10649449,Cloning,lipolytica,squalene,SQS1,erg9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yarrowia lipolytica
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40572208,Heterologous,Nitrogen,Fixing,Cluster,Paenibacillus,polymyxa,subtilis,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Paenibacillus polymyxa
40572208,Heterologous,Nitrogen,Fixing,Cluster,Paenibacillus,polymyxa,subtilis,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Bacillus subtilis
40577193,Structural,AdhE,ultrastructures,Acetivibrio,thermocellus,intermediates,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Acetivibrio thermocellus
40578703,Overcoming,cellobiose,bioconversion,pectin,rich,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Saccharomyces cerevisiae


In [83]:
title_data.loc[:,'Genes'] = title_data.apply(lambda x: get_genes_from_text(x.iloc[0:-1],x.iloc[-1].capitalize(), organism_genes=coso), axis=1)

Organism Chromobacterium viscosum not found
Organism Leuconostoc spp not found
Organism Aspergillus fumigatus not found
Organism Beta proteobacterium not found
Organism Streptomyces spp not found
Organism Candida tropicalis not found
Organism Lactobacillus spp not found
Organism Prochloron spp not found
Organism Nostoc pcc not found
Organism Geobacillus spp not found
Organism Candida tropicalis not found
Organism Variovorax paradoxus not found
Organism Cyanobacteria pcc not found
Organism Synechococcus pcc not found
Organism Candida tropicalis not found
Organism Bacillus spp not found
Organism Sulfolobus spp not found
Organism Synechococcus pcc not found
Organism Thermoanaerobacter spp not found
Organism Synechococcus pcc not found
Organism Candida tropicalis not found
Organism Candida tropicalis not found
Organism Corynebacterium crenatum not found
Organism Thermoanaerobacter spp not found
Organism Corynebacterium crenatum not found
Organism Octopus spring not found
Organism Candida t

Organism Synechococcus pcc not found
Organism Corynebacterium crenatum not found
Organism Halomonas spp not found
Organism Corynebacterium crenatum not found
Organism Medicago spp not found
Organism Streptomyces albulus not found
Organism Candida tropicalis not found
Organism Gracilibacillus alcaliphilus not found
Organism Candida tropicalis not found
Organism Apiotrichum siamense not found
Organism Podospora anserina not found
Organism Methanosarcina spp not found
Organism Phormidium yuhuli not found
Organism Synechococcus pcc not found
Organism Aspergillus fumigatus not found
Organism Bacillus velezensis not found
Organism Streptomyces albulus not found
Organism Aspergillus fumigatus not found
Organism Caldicellulosiruptor morganii not found
Organism Red sea not found
Organism Sporomusa aerivorans not found
Organism Streptomyces albulus not found
Organism Synechococcus pcc not found
Organism Aspergillus fumigatus not found
Organism Candida tropicalis not found
Organism Streptomyces a

In [84]:
abstract_data = classified_org_format_abstract.merge(org_prod.loc[:,['Organism']], left_index=True, right_index=True)

In [90]:
def extend_lists(s):
    result = []
    for lst in s:
        result.extend(lst)
    return result

# Assuming your DataFrame is stored in a variable named `df`
#result = org_prod.dropna(subset=['Genes']).groupby(level=0).agg({'Genes': extend_lists})

# Display the resulting DataFrame
#print(result)

In [95]:
title_data.dropna(subset=['Genes']).groupby(level=0).agg({'Genes': extend_lists})

,Genes
10649449,"[[SQS1, A6ZRL6_P53866A6ZRL6_P53866, sce:YNL224..."
10919795,"[[XKS1, A0A8H4FAD1_P42826A0A8H4FAD1_P42826, sc..."
11004185,"[[LytB, Q55643_Q55763Q55643_Q55763, syn:slr034..."
11418562,"[[PhaP, A0A1K0JNB5_F8GXX9A0A1K0JNB5_F8GXX9, cn..."
11792456,"[[fabA, A0A080J9P8_A0A0E0Y2Y5_A0A0H2YXH9_A0A0H..."
...,...
40428336,"[[fabA, A0A072ZI65_B7UVB1_O33877_Q02K95A0A072Z..."
40447007,"[[SpeE, A0A9P1JKC8_A0A9Q3LFD4A0A9P1JKC8_A0A9Q3..."
40464575,"[[XylR, Q44406, ate:Athe_0617;, 0]]"
40542515,"[[PhzO, A0A125S8Q0, 0, 0]]"


In [96]:
title_data_group = title_data.dropna(subset=['Genes']).groupby(level=0).agg({'Genes': extend_lists})
#org_prod['Genes'].dropna().groupby(level=0).agg(list)
from_title = title_data_group.Genes.dropna().index

In [127]:
from_title_t = title_data_group.Genes.dropna().index

In [128]:
from_title_t

Index([10649449, 10919795, 11004185, 11418562, 11792456, 11841940, 11914362,
       11988503, 12039744, 12828646,
       ...
       40259323, 40277366, 40304514, 40328212, 40343512, 40428336, 40447007,
       40464575, 40542515, 40577193],
      dtype='int64', length=455)

In [129]:
classified_org.loc[from_title_t,'Genes']= title_data_group.loc[:,'Genes']


In [98]:
classified_org.loc[:,'Genes']= title_data_group.loc[:,'Genes']
title_product_count = len(classified_org['Genes'].dropna())
from_title = classified_org.Genes.dropna().index

In [101]:
classified_org

,Title,Abstract,Journal,Year,PMC_ID,DOI,Type,Author,Text,Abstract_min,Title_min,Full_text_min,Old_name,Organism_Source,Organism,Genes
10618204,Increased production of zeaxanthin and other p...,The psbAII locus was used as an integration pl...,Applied and environmental microbiology,2000,91786.0,10.1128/AEM.66.1.64-72.2000,"Journal Article,Research Support, U.S. Gov't, ...","[ForeName:D,LastName:Lagarde] [ForeName:L,Last...",None,psbAII overexpress Synechocystis . PCC psbAII ...,pigments techniques Synechocystis . PCC .,NaN,[Synechocystis PCC],title,[Synechocystis PCC],NaN
10618209,Expression of Alcaligenes eutrophus flavohemop...,Expression of the vhb gene encoding hemoglobin...,Applied and environmental microbiology,2000,91791.0,10.1128/AEM.66.1.98-104.2000,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:A D,LastName:Frey] [ForeName:J E,Las...",None,vhb Vitreoscilla . (VHb) organisms microaerobi...,Alcaligenes eutrophus flavohemoprotein Vitreos...,NaN,[Escherichia coli],title,[Escherichia coli],NaN
10631776,Environmental biotechnology.,There is an increasing interest in environment...,Trends in biotechnology,2000,NaN,10.1016/s0167-7799(99)01399-2,"Journal Article,","[ForeName:L P,LastName:Wackett]",None,"world' maintain soil, water. biology. Plants r...",Environmental biotechnology.,NaN,None,not_found,None,NaN
10649237,Altered regulation of pyruvate kinase or co-ov...,Glycolytic fluxes in resting Escherichia coli ...,Biotechnology and bioengineering,2000,NaN,10.1002/(sici)1097-0290(20000305)67:5<623::aid...,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:M,LastName:Emmerling] [ForeName:J E,...",None,Glycolytic resting kinases (Pyk) stearothermop...,Altered co- phosphofructokinase glycolytic res...,NaN,[Escherichia coli],title,[Escherichia coli],NaN
10649449,Cloning and characterization of the Yarrowia l...,The squalene synthase (SQS) gene encodes a key...,"Yeast (Chichester, England)",2000,NaN,10.1002/(SICI)1097-0061(200002)16:3<197::AID-Y...,"Journal Article,Research Support, Non-U.S. Gov...","[ForeName:S,LastName:Merkulov] [ForeName:F,Las...",None,"squalene (SQS) encodes , farnesyl-diphosphate ...",Cloning lipolytica squalene (SQS1) erg9 .,NaN,"[Yarrowia lipolytica, Saccharomyces cerevisiae]",title,"[Saccharomyces cerevisiae, Yarrowia lipolytica]","[[SQS1, A6ZRL6_P53866A6ZRL6_P53866, sce:YNL224..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40572208,Heterologous Expression of the Nitrogen-Fixing...,Microbially mediated biological nitrogen fixat...,Microorganisms,2025,12195393.0,10.3390/microorganisms13061320,"Journal Article,","[ForeName:Xiuling,LastName:Wang] [ForeName:Shi...",Introduction:\nThis article reports the first ...,"Microbially . , hindered complexities manipula...",Heterologous Nitrogen-Fixing Cluster <i>Paenib...,Introduction: reports nif .polymyxa .subtilis....,"[Paenibacillus polymyxa, Bacillus subtilis]",title,"[Paenibacillus polymyxa, Bacillus subtilis]",NaN
40573728,Functional Characterization of Squalene Epoxid...,The medicinal plant <i>Siraitia grosvenorii</i...,"Plants (Basel, Switzerland)",2025,NaN,10.3390/plants14121740,"Journal Article,","[ForeName:Huan,LastName:Zhao] [ForeName:Ze,Las...",None,<i>Siraitia grosvenorii</i> sweet-tasting cucu...,Functional Characterization Squalene Epoxidase...,NaN,None,abstract,None,NaN
40577193,Structural characterization and dynamics of Ad...,"<i>Acetivibrio thermocellus</i>, a cellulolyti...",eLife,2025,NaN,10.7554/eLife.96966,"Journal Article,","[ForeName:Samantha J,LastName:Ziegler] [ForeNa...",None,"<i>Acetivibrio thermocellus</i>, cellulolytic ...",Structural AdhE ultrastructures <i>Acetivibrio...,NaN,[Acetivibrio thermocellus],title,[Acetivibrio thermocellus],"[[AdhE, A0A0H3W5I3, 0, 0]]"
40578703,Overcoming glucose repression through cellobio...,Pectin-rich biomass is a promising substrate f...,Bioresource technology,2025,NaN,10.1016/j.biortech.2025.132892,"Journal Article,","[ForeName:Dahye,LastName:Lee] [

Change to not search if the organism is not in the list, it makes no sense to search if we already know we dont have that info :( 

In [103]:
articles_no_genes_from_title = classified_org.loc[classified_org['Genes'].isna()].index
abstract_data_no_genes_from_title = abstract_data.loc[abstract_data.index.isin(articles_no_genes_from_title)]
result_absta = abstract_data_no_genes_from_title.apply(lambda x: get_genes_from_text(x.iloc[0:-1],x.iloc[-1].capitalize(), organism_genes=coso), axis=1)

Organism Chromobacterium viscosum not found
Organism Leuconostoc spp not found
Organism Aspergillus fumigatus not found
Organism Beta proteobacterium not found
Organism Streptomyces spp not found
Organism Candida tropicalis not found
Organism Lactobacillus spp not found
Organism Prochloron spp not found
Organism Nostoc pcc not found
Organism Geobacillus spp not found
Organism Candida tropicalis not found
Organism Variovorax paradoxus not found
Organism Cyanobacteria pcc not found
Organism Synechococcus pcc not found
Organism Candida tropicalis not found
Organism Bacillus spp not found
Organism Sulfolobus spp not found
Organism Synechococcus pcc not found
Organism Thermoanaerobacter spp not found
Organism Synechococcus pcc not found
Organism Candida tropicalis not found
Organism Candida tropicalis not found
Organism Corynebacterium crenatum not found
Organism Thermoanaerobacter spp not found
Organism Corynebacterium crenatum not found
Organism Octopus spring not found
Organism Candida t

Organism Corynebacterium crenatum not found
Organism Halomonas spp not found
Organism Corynebacterium crenatum not found
Organism Medicago spp not found
Organism Streptomyces albulus not found
Organism Candida tropicalis not found
Organism Gracilibacillus alcaliphilus not found
Organism Candida tropicalis not found
Organism Apiotrichum siamense not found
Organism Podospora anserina not found
Organism Methanosarcina spp not found
Organism Phormidium yuhuli not found
Organism Synechococcus pcc not found
Organism Aspergillus fumigatus not found
Organism Bacillus velezensis not found
Organism Streptomyces albulus not found
Organism Aspergillus fumigatus not found
Organism Caldicellulosiruptor morganii not found
Organism Red sea not found
Organism Sporomusa aerivorans not found
Organism Streptomyces albulus not found
Organism Synechococcus pcc not found
Organism Aspergillus fumigatus not found
Organism Candida tropicalis not found
Organism Streptomyces albulus not found
Organism Bacillus ve

In [108]:
result_abstract.name='Genes'
result_abstract = pd.DataFrame(result_abstract)
result_abstract_group = result_abstract.dropna(subset=['Genes']).groupby(level=0).agg({'Genes': extend_lists})

from_abstract = result_abstract_group.Genes.dropna().index

In [130]:
classified_org.loc[from_abstract,'Genes']= result_abstract_group.loc[:,'Genes']
#title_product_count = len(classified_org['Genes'].dropna())
#from_title = classified_org.Genes.dropna().index

In [119]:
articles_no_genes_from_title_abstract = classified_org.loc[classified_org['Genes'].isna()].index
fulltext_data_no_genes_from_title_abstract = fulltext_data.loc[fulltext_data.index.isin(articles_no_genes_from_title_abstract)]
result_full_text = fulltext_data_no_genes_from_title_abstract.apply(lambda x: get_genes_from_text(x.iloc[0:-1],x.iloc[-1].capitalize(), organism_genes=coso), axis=1)

Organism Leuconostoc spp not found
Organism Aspergillus fumigatus not found
Organism Streptomyces spp not found
Organism Candida tropicalis not found
Organism Variovorax paradoxus not found
Organism Synechococcus pcc not found
Organism Candida tropicalis not found
Organism Bacillus spp not found
Organism Sulfolobus spp not found
Organism Synechococcus pcc not found
Organism Corynebacterium crenatum not found
Organism Chromohalobacter salexigens not found
Organism Desulfonema limicola not found
Organism Aspergillus fumigatus not found
Organism Streptomyces spp not found
Organism Chromohalobacter salexigens not found
Organism Vitis spp not found
Organism Streptomyces spp not found
Organism Anabaena spp not found
Organism Synechococcus pcc not found
Organism Synechococcus pcc not found
Organism Nocardia tartaricans not found
Organism Synechococcus pcc not found
Organism Thermotoga spp not found
Organism Synechococcus pcc not found
Organism Synechococcus pcc not found
Organism Aspergillus 

In [120]:
result_full_text.name='Genes'
result_full_text = pd.DataFrame(result_full_text)
result_full_text_group = result_full_text.dropna(subset=['Genes']).groupby(level=0).agg({'Genes': extend_lists})

from_full_text = result_full_text_group.Genes.dropna().index

In [124]:
from_title

Index([10649449, 10919795, 11004185, 11418562, 11792456, 11841940, 11914362,
       11988503, 12039744, 12828646,
       ...
       40259323, 40277366, 40304514, 40328212, 40343512, 40428336, 40447007,
       40464575, 40542515, 40577193],
      dtype='int64', length=455)

In [126]:
set(from_abstract).intersection(set(from_full_text))

set()

In [122]:
from_full_text

Index([10662693, 10742205, 10835112, 10931852, 10972798, 11004185, 11048953,
       11097884, 11118589, 11137817,
       ...
       40510534, 40510666, 40529627, 40537803, 40542515, 40544235, 40552110,
       40558924, 40564374, 40572067],
      dtype='int64', length=4798)

In [131]:
classified_org.loc[from_full_text,'Genes']= result_full_text_group.loc[:,'Genes']
#title_product_count = len(classified_org['Genes'].dropna())
#from_title = classified_org.Genes.dropna().index

In [132]:
classified_org.Genes.dropna()

10618204    [[crtB, P37294, syn:slr1255;, 2.5.1.32;], [crt...
10649449    [[SQS1, A6ZRL6_P53866A6ZRL6_P53866, sce:YNL224...
10653745                           [[phaC, A0A3L0W3F5, 0, 0]]
10662693    [[eryAI, A4F7N8_O33937_Q5UNP6, sen:SACE_0721;,...
10742205    [[der, A0A023KS59_A0A0E0XX99_A0A0H2Z1F8_A0A0H3...
                                  ...                        
40564374    [[ATCC, F8GVS8, cnc:CNE_BB1p01430;, 3.5.1.87;]...
40572067    [[amyL, L7X1Q1_P06278_Q65MX0L7X1Q1_P06278_Q65M...
40572208    [[veg, E3EDV4, ppm:PPSC2_00160;, 0], [veg, D4G...
40577193                           [[AdhE, A0A0H3W5I3, 0, 0]]
40579636                             [[ccpA, A0A345FZU8, , ]]
Name: Genes, Length: 8885, dtype: object

In [140]:
from_title_t.drop_duplicates()

Index([10649449, 10919795, 11004185, 11418562, 11792456, 11841940, 11914362,
       11988503, 12039744, 12828646,
       ...
       40259323, 40277366, 40304514, 40328212, 40343512, 40428336, 40447007,
       40464575, 40542515, 40577193],
      dtype='int64', length=455)

In [145]:
set(from_full_text).intersection(set(from_title_t))

{11004185,
 11418562,
 11841940,
 11914362,
 12039744,
 12828646,
 14529965,
 15184121,
 16204475,
 16391053,
 16597951,
 16820460,
 16865736,
 17369345,
 17688423,
 18040680,
 18667064,
 18791032,
 19047353,
 19296857,
 19617392,
 20102600,
 20213524,
 20214823,
 20224601,
 20521041,
 20544254,
 20693441,
 21267412,
 21272324,
 21375718,
 21478306,
 21819557,
 22008943,
 22056446,
 22374228,
 22486967,
 22572787,
 22851018,
 22892576,
 22978798,
 23053109,
 23060871,
 23064346,
 23149679,
 23263959,
 23398717,
 23422309,
 23448319,
 23475614,
 23503313,
 23526182,
 23892063,
 24005667,
 24174215,
 24190496,
 24212572,
 24244597,
 24360128,
 24733517,
 24904688,
 24932218,
 25012491,
 25013760,
 25147754,
 25181035,
 25229866,
 25258165,
 25278273,
 25298782,
 25359316,
 25363722,
 25368804,
 25420425,
 25441601,
 25450012,
 25482228,
 25666131,
 25743068,
 25845305,
 25852989,
 25861759,
 25889168,
 25981549,
 26013492,
 26052021,
 26159300,
 26302366,
 26509553,
 26539179,
 26666990,

In [144]:
set(from_full_text).intersection(set(from_abstract))

set()

In [143]:
set(from_title_t).intersection(set(from_abstract))

set()

In [141]:
set(from_abstract).intersection(set(from_title_t))

set()

In [137]:
classified_org['Gene_Source'] = 'not_found'
classified_org.loc[from_title_t, 'Gene_Source'] = 'title'
classified_org.loc[from_abstract, 'Gene_Source'] = 'abstract'
classified_org.loc[from_full_text, 'Gene_Source'] = 'full_text'


In [146]:
len(from_full_text)-len(set(from_full_text).intersection(set(from_title_t)))

4559

In [138]:
classified_org.Gene_Source.value_counts()

Gene_Source
not_found    7244
full_text    4798
abstract     3871
title         216
Name: count, dtype: int64

In [136]:
check_save_file(classified_org, output_file, 'Articles')

Saved file in: /Users/elisamarquez/Documents/PhD/2semestre/Engineering_db/Engineering_db/files/Output/Articles/full_text_w_genes_25_09_26_new_v.json
